# Nested transformer optimization study

This notebook runs only the remaining encoder, hyperparameter, and seed-stability study. It does not rerun the already published DistilBERT and Twitter-RoBERTa fixed benchmarks. Use a GPU runtime and run one outer fold per session.

In [ ]:
!git clone --depth 1 https://github.com/kristishqau/SentimentAnalysis_NLP.git
%cd SentimentAnalysis_NLP
%pip install -q -e ".[transformer]" "scikit-learn==1.9.1"

In [ ]:
import torch

assert torch.cuda.is_available(), "Select Runtime -> Change runtime type -> GPU"
print(torch.cuda.get_device_name(0))

## Run and immediately download one fold

Set `FOLD_TO_RUN` to 1, then repeat in later sessions with 2, 3, 4, and 5. Each fold performs the complete four-encoder screen, bounded tuning search, and three-seed stability evaluation.

In [ ]:
import subprocess

from google.colab import files

FOLD_TO_RUN = 1
assert FOLD_TO_RUN in range(1, 6)
output = f"artifacts/transformer_study_fold_{FOLD_TO_RUN}.json"
subprocess.run(
    [
        "sentiment-transformer-study",
        "--outer-fold",
        str(FOLD_TO_RUN),
        "--epochs",
        "4",
        "--batch-size",
        "32",
        "--reference",
        "reports/benchmark_v2.json",
        "--output",
        output,
    ],
    check=True,
)
files.download(output)

## Merge after all five folds

Start a fresh session, run the setup cells, then select and upload all five downloaded fold files. The strict merger rejects missing, duplicate, or incompatible evidence.

In [ ]:
uploaded = files.upload()
expected = [f"transformer_study_fold_{fold}.json" for fold in range(1, 6)]
assert sorted(uploaded) == sorted(expected), f"Expected exactly: {expected}"
subprocess.run(
    [
        "sentiment-transformer-study-merge",
        *expected,
        "--output",
        "reports/transformer_optimization_study.json",
    ],
    check=True,
)
files.download("reports/transformer_optimization_study.json")